# DEMO: Creating and Querying the Different Types of Views

Views are the direct replacement for Power Query transformations in Databricks. Instead of building steps in a visual editor, you write a SQL query and save it as a **view** in Unity Catalog. Anyone with permission can then query it.

In this demo, you’ll learn:
- **Standard Views** — saved SQL that runs fresh every time (most common)
- **Temporary Views** — session-scoped, disappear when you disconnect
- **Materialized Views** — precomputed results, stored physically for performance

---

In [0]:
%run ./Setup

## Our Data

The setup created a `raw_orders` table with 500 orders. This simulates messy source data:
- Some orders are `completed`, some are `pending` or `cancelled`
- Has raw pricing columns (`quantity`, `unit_price`, `discount_pct`) that need calculation
- Contains a `customer_email` column (PII we might want to hide)

In the real world, you wouldn’t want dashboard users querying this raw table directly. Instead, you’d create **views** that present clean, transformed data.

---

In [0]:
%sql
-- Let's look at the raw source data
SELECT * FROM raw_orders LIMIT 5;

## Part 1: Standard Views

A **standard view** is a saved SQL query stored permanently in Unity Catalog. Every time someone queries the view, the underlying SQL executes against the latest data.

```sql
CREATE OR REPLACE VIEW view_name AS
SELECT ... FROM ... WHERE ...;
```

| Feature | Detail |
| --- | --- |
| **Persistence** | Permanent — stored in Unity Catalog, visible to all with permission |
| **Freshness** | Always returns the latest data (SQL re-executes every time) |
| **Storage** | No data stored — only the SQL definition is saved |
| **Governance** | Governed by Unity Catalog permissions (GRANT/REVOKE) |

**When to use:** Most of the time! Views are your default choice for reusable transformations, clean semantic layers for dashboards, and access control (hide raw columns like PII).

---

In [0]:
%sql
-- ============================================================
-- Standard View: Clean, completed orders with calculated revenue
-- ============================================================
-- This view does three things:
-- 1. Filters to only completed orders (hides pending/cancelled)
-- 2. Calculates line_total (the business logic lives here)
-- 3. Hides PII (customer_email is not included)

CREATE OR REPLACE VIEW v_completed_orders AS
SELECT 
  order_id,
  order_date,
  region,
  product_category,
  quantity,
  unit_price,
  discount_pct,
  ROUND(quantity * unit_price * (1 - discount_pct / 100.0), 2) AS line_total
FROM raw_orders
WHERE status = 'completed';

In [0]:
%sql
-- Now anyone can query the view as if it were a table
-- They don't need to know the filtering or calculation logic
SELECT 
  region,
  COUNT(*) AS orders,
  ROUND(SUM(line_total), 2) AS total_revenue,
  ROUND(AVG(line_total), 2) AS avg_order_value
FROM v_completed_orders
GROUP BY region
ORDER BY total_revenue DESC;

In [0]:
%sql
-- ============================================================
-- Standard View: Daily revenue summary (a "Gold" view)
-- ============================================================
-- This builds on v_completed_orders to create a daily aggregate.
-- Views can reference other views!

CREATE OR REPLACE VIEW v_daily_revenue_summary AS
SELECT 
  order_date,
  region,
  product_category,
  COUNT(*) AS order_count,
  ROUND(SUM(line_total), 2) AS daily_revenue,
  ROUND(AVG(line_total), 2) AS avg_order_value
FROM v_completed_orders
GROUP BY order_date, region, product_category;

In [0]:
%sql
-- Query the layered view - perfect for a dashboard
SELECT 
  order_date,
  SUM(daily_revenue) AS total_daily_revenue
FROM v_daily_revenue_summary
WHERE region = 'North'
GROUP BY order_date
ORDER BY order_date
LIMIT 10;

## Views Always Reflect the Latest Data

Unlike a Power BI Import-mode dataset (which shows data as of the last refresh), a standard view always queries the **live source data**. Let’s prove it:

---

In [0]:
%sql
-- Step 1: Check current count
SELECT COUNT(*) AS order_count FROM v_completed_orders;

In [0]:
%sql
-- Step 2: Insert a new completed order into the source table
INSERT INTO raw_orders VALUES (
  9999, DATE'2024-03-30', 'North', 'Electronics', 'completed', 
  5, 99.99, 0, 'new_customer@example.com'
);

In [0]:
%sql
-- Step 3: Query the view again - it immediately reflects the new row!
-- No refresh needed. The view's SQL re-executes against the live table.
SELECT COUNT(*) AS order_count FROM v_completed_orders;

## Part 2: Temporary Views

A **temporary view** exists only for the duration of your current session. Once you disconnect (or the cluster restarts), it’s gone.

```sql
CREATE OR REPLACE TEMPORARY VIEW tv_name AS
SELECT ... FROM ...;
```

| Feature | Detail |
| --- | --- |
| **Persistence** | Session only — disappears when you disconnect |
| **Visibility** | Only visible to your session (not shared with others) |
| **Storage** | Not stored in Unity Catalog |
| **Governance** | Not governed — no GRANT/REVOKE |
| **Power BI analogy** | A Power Query step you haven’t published yet |

**When to use:** For ad-hoc exploration, intermediate calculations within a notebook session, or when you need a named result temporarily but don’t want to clutter the catalog.

---

In [0]:
%sql
-- ============================================================
-- Temporary View: session-scoped, for ad-hoc exploration
-- ============================================================
-- This view only exists for THIS session.
-- It won't be visible to other users or in Catalog Explorer.

CREATE OR REPLACE TEMPORARY VIEW tv_top_categories AS
SELECT 
  product_category,
  COUNT(*) AS order_count,
  ROUND(SUM(quantity * unit_price * (1 - discount_pct / 100.0)), 2) AS revenue
FROM raw_orders
WHERE status = 'completed'
GROUP BY product_category
ORDER BY revenue DESC;

In [0]:
%sql
-- Query the temp view (works just like a regular view)
SELECT * FROM tv_top_categories;

In [0]:
%sql
-- ============================================================
-- SHOW VIEWS reveals both standard and temporary views
-- ============================================================
-- Notice the isTemporary column: standard views show FALSE,
-- the temp view shows TRUE and has no namespace (not in catalog)

SHOW VIEWS;

## Part 3: Materialized Views

A **materialized view** precomputes and physically stores the query results. Unlike a standard view (which re-runs its SQL every time), a materialized view returns the stored results instantly, then refreshes on a trigger or schedule.

```sql
CREATE MATERIALIZED VIEW mv_name AS
SELECT ... FROM ...;
```

| Feature | Detail |
| --- | --- |
| **Persistence** | Permanent — stored in Unity Catalog with physical data |
| **Freshness** | Results are precomputed; refreshes on trigger, schedule, or manual refresh |
| **Storage** | Data IS physically stored (uses storage space) |
| **Performance** | Much faster queries (reads stored results, not source) |
| **Power BI analogy** | An Import-mode dataset with scheduled refresh |

**When to use:** When a view query is too slow for interactive dashboards (typically 30+ seconds), or when the same expensive aggregation is queried repeatedly by many users.

**Trade-off:** Faster reads, but data might be slightly stale between refreshes — exactly like the Import vs DirectQuery trade-off in Power BI.

## Materialized View vs. Managed Table

If a materialized view physically stores data, you might ask: how is it different from just creating a managed table and refreshing it with a scheduled job?

The answer is **who manages the refresh**. With a managed table you own everything — you write the ETL (`INSERT OVERWRITE`, `MERGE`, etc.), schedule a job, and decide what gets recomputed. The table is just a data container.

With a materialized view, the SQL query IS the definition, and Databricks owns the refresh behavior:

| | Materialized View | Managed Table |
| --- | --- | --- |
| **Refresh trigger** | Automatic (upstream change, schedule, or manual) | You write + schedule the ETL yourself |
| **Incremental processing** | Databricks can refresh only changed rows | Full recompute unless you write MERGE logic |
| **Lineage** | Automatically tracked in Unity Catalog | Not tracked automatically |
| **Stale detection** | Databricks knows when results are out of date | You manage this yourself |
| **Source of truth** | The SQL definition | Whatever data was last written |

In short: a materialized view is a managed table where **Databricks writes and maintains the ETL for you**.

> **Important:** Run the materialized view examples in the **SQL Editor on a SQL Warehouse**. They will not work reliably from this notebook's serverless compute.

---

## Run These Materialized View Queries in the SQL Editor

For this part of the demo, switch from the notebook to the **SQL Editor**.

### Why?

Materialized views are created and refreshed on a **SQL Warehouse**. Notebook serverless compute is great for setup and mixed-language workflows, but it is not the right place to run these materialized view statements.

### What to do:

1. In the left sidebar, open **SQL Editor**
2. Create a new **SQL Query**
3. Select a running **SQL Warehouse**
4. Set your default **Catalog** and **Schema**
5. Choose the schema that starts with `demo_views_` — that is the schema created by the setup notebook
6. Copy the materialized view queries from the cells below into the SQL Editor and run them there

> Tip: You can still use this notebook for setup and explanation, but run the materialized view SQL itself in the SQL Editor, following the same pattern as the [Demo](#notebook-3244242923465736) notebook in the [SQL_Editor](#folder-3244242923465684) folder.


```
-- ============================================================
-- Materialized View: precomputed results for fast queries
-- ============================================================
-- Run this in the SQL Editor on a SQL Warehouse.
-- This stores the aggregated results physically.
-- Subsequent queries read the stored results, not the raw table.

CREATE OR REPLACE MATERIALIZED VIEW mv_regional_summary AS
SELECT 
  region,
  product_category,
  COUNT(*) AS total_orders,
  SUM(CASE WHEN status = 'completed' THEN 1 ELSE 0 END) AS completed_orders,
  ROUND(SUM(CASE WHEN status = 'completed' THEN quantity * unit_price * (1 - discount_pct / 100.0) ELSE 0 END), 2) AS total_revenue
FROM raw_orders
GROUP BY region, product_category;
```

## Materialized View Refresh Strategies

Once a materialized view is created, it holds a snapshot of the data at creation time. You need to refresh it to pick up changes from the source. There are four ways to control when that happens:

| Strategy | Syntax Example | How It Works |
| --- | --- | --- |
| **Trigger on Update** | `TRIGGER ON UPDATE AT MOST EVERY INTERVAL 1 HOUR` | Databricks watches the upstream source and refreshes automatically when new data arrives. The `AT MOST EVERY` throttle prevents runaway refreshes if data changes very frequently. |
| **Fixed Schedule** | `SCHEDULE EVERY 1 HOUR` | Refreshes on a fixed clock interval regardless of whether the source actually changed. Supports `HOUR`, `DAY`, and `WEEK` — does **not** support minutes. |
| **Cron Schedule** | `SCHEDULE CRON '0 0 6 ? * MON-FRI *' AT TIME ZONE 'Australia/Sydney'` | Uses a Quartz cron expression for precise or complex timing (e.g. weekdays at 6 AM). Use this when `SCHEDULE EVERY` is not flexible enough. |
| **Manual Refresh** | `REFRESH MATERIALIZED VIEW mv_name` | Triggered on demand. Useful for one-off refreshes, testing, or backfill scenarios. |

---

### Why `TRIGGER ON UPDATE` Matters

`TRIGGER ON UPDATE` is often the best default refresh strategy for a materialized view.

Instead of refreshing every hour whether the source changed or not, Databricks watches for upstream updates and refreshes only when needed. The `AT MOST EVERY INTERVAL 1 HOUR` part prevents refreshes from happening too frequently.

This is usually better than a fixed schedule when your source data arrives irregularly.



```
-- ============================================================
-- Materialized View with TRIGGER ON UPDATE
-- ============================================================
-- Recommended when you want Databricks to refresh automatically
-- when upstream data changes, without wasting refreshes.
--
-- Run this in the SQL Editor on a SQL Warehouse.

CREATE OR REPLACE MATERIALIZED VIEW mv_regional_summary_triggered
TRIGGER ON UPDATE AT MOST EVERY INTERVAL 1 HOUR
AS
SELECT 
  region,
  product_category,
  COUNT(*) AS total_orders,
  SUM(CASE WHEN status = 'completed' THEN 1 ELSE 0 END) AS completed_orders,
  ROUND(SUM(CASE WHEN status = 'completed' THEN quantity * unit_price * (1 - discount_pct / 100.0) ELSE 0 END), 2) AS total_revenue
FROM raw_orders
GROUP BY region, product_category;
```

## Inspecting View Definitions

You can always check what SQL a view is built from using `SHOW CREATE TABLE` (yes, it works for views too):

---

In [0]:
%sql
-- See the SQL definition behind a view
SHOW CREATE TABLE v_completed_orders;

In [0]:
%sql
-- Describe works on views just like tables
DESCRIBE TABLE EXTENDED v_completed_orders;

---

## Summary: Choosing the Right View Type

| View Type | Create Syntax | Persistence | Performance | Best For |
| --- | --- | --- | --- | --- |
| **Standard View** | `CREATE VIEW` | Permanent (Unity Catalog) | Re-executes each query | Most transformations, semantic layers, access control |
| **Temporary View** | `CREATE TEMPORARY VIEW` | Session only | Re-executes each query | Ad-hoc exploration, intermediate results |
| **Materialized View** | `CREATE MATERIALIZED VIEW` | Permanent (physically stored) | Reads cached results | Expensive aggregations queried frequently |

**Decision guide:**
1. **Start with a standard view** — it’s simple, always fresh, and governed
2. If the view query takes **30+ seconds** and is queried by many users, consider a materialized view
3. Use temp views for **scratchpad work** within a session